# Clasificación del Dataset Iris con K-NN

Este ejercicio práctico muestra el flujo de trabajo para clasificar el dataset de flores Iris utilizando el algoritmo k-Nearest Neighbors (k-NN), incluyendo la división de datos, evaluación y la búsqueda del valor óptimo de $k$ mediante el **Método del Codo**.

## 1. Importación de Librerías y Carga de Datos

Utilizaremos las herramientas de `scikit-learn` para cargar el dataset Iris.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Cargamos el dataset
iris = load_iris()
X = iris.data
y = iris.target

print("Características:", iris.feature_names)
print("Clases:", iris.target_names)

## 2. Preparación del Conjunto de Datos

Siguiendo los requerimientos, dividiremos los datos en:
- **80% Entrenamiento**
- **20% Prueba**

> **Nota:** Es fundamental escalar los datos en k-NN porque este algoritmo se basa en distancias. Si una característica tiene una escala mucho mayor que otra, dominará el cálculo de la distancia.

In [ ]:
# División 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Escalado de características (Estandarización)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"X_train: {len(X_train)} muestras")
print(f"X_test: {len(X_test)} muestras")

## 3. Clasificación con K-NN Inicial

Probaremos un valor inicial (por ejemplo, $k=5$) para ver el desempeño base.

In [ ]:
knn_inicial = KNeighborsClassifier(n_neighbors=5)
knn_inicial.fit(X_train, y_train)
y_pred = knn_inicial.predict(X_test)

print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred))
print("Exactitud (Accuracy):", accuracy_score(y_test, y_pred))

## 4. Investigación: ¿Qué es el Método del Codo?

El **Método del Codo (Elbow Method)** es una técnica visual que se utiliza para elegir el valor óptimo de un hiperparámetro (como $k$ en k-NN o el número de clusters en K-Means).

### En el contexto de k-NN:
Consiste en entrenar el modelo repetidas veces variando el valor de $k$ y graficar una métrica de error (como la **Tasa de Error**) frente a cada $k$.

- **Objetivo:** Identificar el punto donde el error disminuye drásticamente y luego comienza a estabilizarse o incluso a subir (debido al sobreajuste).
- **Interpretación:** Ese punto de inflexión en la gráfica se asemeja a un 'codo'. El valor de $k$ en ese punto es usualmente el más equilibrado entre sesgo y varianza.

## 5. Implementación del Método del Codo para el Dataset Iris

Probaremos valores de $k$ desde 1 hasta 40 para observar cómo cambia el error.

In [ ]:
error_rate = []

# Iteramos para diferentes valores de k
for i in range(1, 40):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    # Calculamos el promedio de predicciones incorrectas
    error_rate.append(np.mean(pred_i != y_test))

# Graficamos la tasa de error
plt.figure(figsize=(10,6))
plt.plot(range(1, 40), error_rate, color='blue', linestyle='dashed', marker='o', 
         markerfacecolor='red', markersize=10)
plt.title('Tasa de Error vs. Valor de K')
plt.xlabel('K')
plt.ylabel('Tasa de Error')
plt.grid(True)
plt.show()

## 6. Determinación del valor de $k$ adecuado

Tras observar la gráfica anterior, podemos determinar el valor óptimo.

**Análisis de Resultados:**
- Si la tasa de error es 0 para varios valores, podemos elegir el $k$ más pequeño dentro de la zona estable para mantener la simplicidad del modelo.
- Para el dataset Iris (con 150 muestras), un valor de $k$ entre **3 y 11** suele ser óptimo.
- Evitamos $k=1$ porque suele causar sobreajuste (overfitting) al seguir demasiado de cerca el ruido de los datos.

**Conclusión:**
Basado en la gráfica, el valor de $k$ que recomendaría es aquel donde la tasa de error llega a su primer mínimo estable. (Por ejemplo, si la gráfica muestra un error mínimo a partir de $k=3$ y se mantiene estable, $k=3$ o $k=5$ son excelentes opciones).